# Multi-Agent Customer Support System

## Scenario
You are building a **multi-agent customer support system** for an e-commerce company.  
The business wants faster support, consistent resolution quality, and cleaner escalation handling.

Instead of one chatbot handling everything, the system is split into multiple agents:

1. **Triage Agent** -> Understands the issue type  
2. **Order Agent** -> Checks order details  
3. **Policy Agent** -> Applies refund / return / delay rules  
4. **Escalation Agent** -> Decides whether a human ticket is needed  
5. **Response Agent** -> Writes the final customer-facing message  

## What this notebook includes
- **Groq API integration**
- **FastAPI endpoints**
- **Multi-agent orchestration**
- **Mock order database + ticket database**
- **Gradio UI for Colab demo**
- **Heavy comments for understanding and viva explanation**

In [ ]:
# Install required packages.
!pip install -q groq fastapi uvicorn gradio pandas nest_asyncio

In [ ]:
# ============================================================
# IMPORTS + API KEY SETUP
# ============================================================

import os
import json
import re
from datetime import datetime
from typing import Dict, Any, List, Optional

import pandas as pd
import gradio as gr
import nest_asyncio

from fastapi import FastAPI
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

from groq import Groq

nest_asyncio.apply()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    except Exception:
        GROQ_API_KEY = None

client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
MODEL_NAME = "llama-3.1-8b-instant"

print("Groq client ready." if client else "Groq API key not found yet. Add GROQ_API_KEY in Colab secrets.")

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def extract_json_block(text: str) -> Dict[str, Any]:
    """
    Extract JSON from LLM output safely.
    """
    if not text:
        return {}

    cleaned = text.strip().replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    match = re.search(r"\{.*\}", cleaned, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return {}

    return {}


def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.2) -> str:
    """
    Single wrapper for Groq calls.
    """
    if client is None:
        raise ValueError(
            "Groq client is not initialized. Please add GROQ_API_KEY in Colab secrets or environment variables."
        )

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content

In [ ]:
# ============================================================
# MOCK DATABASES
# ============================================================
# These local stores make the project demo-ready without depending on a real company database.

ORDERS_DB = {
    "ORD1001": {
        "customer_name": "Aarav",
        "status": "delivered",
        "days_since_delivery": 3,
        "item": "Wireless Headphones",
        "payment_status": "paid",
        "issue_notes": ""
    },
    "ORD1002": {
        "customer_name": "Riya",
        "status": "shipped",
        "days_since_delivery": 0,
        "item": "Bluetooth Speaker",
        "payment_status": "paid",
        "issue_notes": ""
    },
    "ORD1003": {
        "customer_name": "Kabir",
        "status": "processing",
        "days_since_delivery": 0,
        "item": "Laptop Stand",
        "payment_status": "pending",
        "issue_notes": ""
    },
}

POLICY_DB = {
    "return_window_days": 7,
    "refund_window_days": 7,
    "damaged_item_rule": "Eligible for priority replacement or refund",
    "late_delivery_rule": "If delay is confirmed, apologize and offer shipment update",
    "payment_failure_rule": "Ask customer to retry payment or use another method"
}

TICKETS_DB = []

In [ ]:
# ============================================================
# TOOL FUNCTIONS
# ============================================================
# These behave like internal tools that specialized agents can call.

def find_order(order_id: str) -> Optional[Dict[str, Any]]:
    return ORDERS_DB.get(order_id)


def create_ticket(issue_type: str, priority: str, customer_message: str, order_id: str = "") -> Dict[str, Any]:
    ticket_id = f"TKT{1000 + len(TICKETS_DB) + 1}"
    ticket = {
        "ticket_id": ticket_id,
        "issue_type": issue_type,
        "priority": priority,
        "order_id": order_id,
        "customer_message": customer_message,
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    TICKETS_DB.append(ticket)
    return ticket


def evaluate_policy(issue_type: str, order: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Applies deterministic business rules before the LLM writes a response.
    This is important because policy logic should be consistent.
    """
    if issue_type == "return_request":
        if order and order["status"] == "delivered" and order["days_since_delivery"] <= POLICY_DB["return_window_days"]:
            return {"eligible": True, "decision": "Return allowed within policy window"}
        return {"eligible": False, "decision": "Return not allowed under current policy window"}

    if issue_type == "refund_request":
        if order and order["status"] == "delivered" and order["days_since_delivery"] <= POLICY_DB["refund_window_days"]:
            return {"eligible": True, "decision": "Refund can be processed"}
        return {"eligible": False, "decision": "Refund not allowed under current policy window"}

    if issue_type == "late_delivery":
        return {"eligible": True, "decision": POLICY_DB["late_delivery_rule"]}

    if issue_type == "payment_issue":
        return {"eligible": True, "decision": POLICY_DB["payment_failure_rule"]}

    if issue_type == "damaged_item":
        return {"eligible": True, "decision": POLICY_DB["damaged_item_rule"]}

    return {"eligible": True, "decision": "General support flow"}

In [ ]:
# ============================================================
# AGENT 1: TRIAGE AGENT
# ============================================================

def heuristic_triage(customer_message: str) -> str:
    """
    Fast fallback classifier using keywords.
    """
    text = customer_message.lower()

    if "refund" in text:
        return "refund_request"
    if "return" in text:
        return "return_request"
    if "late" in text or "delay" in text or "not delivered" in text:
        return "late_delivery"
    if "payment" in text or "paid twice" in text or "transaction" in text:
        return "payment_issue"
    if "damaged" in text or "broken" in text:
        return "damaged_item"

    return "general_query"


def triage_agent(customer_message: str) -> Dict[str, Any]:
    """
    Uses LLM for richer classification, but still keeps a deterministic backup.
    """
    fallback_type = heuristic_triage(customer_message)

    system_prompt = (
        "You are a support triage classifier. "
        "Return only valid JSON with keys: issue_type, priority, sentiment, escalation_needed."
    )

    user_prompt = f"""
    Customer message:
    {customer_message}

    Use one of these issue types:
    refund_request, return_request, late_delivery, payment_issue, damaged_item, general_query
    """

    try:
        llm_text = call_llm(system_prompt, user_prompt, temperature=0.1)
        llm_json = extract_json_block(llm_text)

        issue_type = llm_json.get("issue_type", fallback_type)
        priority = llm_json.get("priority", "medium")
        sentiment = llm_json.get("sentiment", "neutral")
        escalation_needed = llm_json.get("escalation_needed", False)

        return {
            "issue_type": issue_type,
            "priority": priority,
            "sentiment": sentiment,
            "escalation_needed": escalation_needed,
            "source": "llm+heuristic"
        }
    except Exception:
        return {
            "issue_type": fallback_type,
            "priority": "medium",
            "sentiment": "neutral",
            "escalation_needed": False,
            "source": "heuristic_only"
        }

In [ ]:
# ============================================================
# AGENT 2 / 3 / 4 / 5
# ============================================================

def order_agent(order_id: str) -> Dict[str, Any]:
    order = find_order(order_id)
    return {"order_found": order is not None, "order": order}


def policy_agent(issue_type: str, order: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    return evaluate_policy(issue_type, order)


def escalation_agent(issue_type: str, priority: str, policy_result: Dict[str, Any]) -> Dict[str, Any]:
    """
    Decide when to escalate to human support.
    """
    needs_ticket = False

    if priority.lower() == "high":
        needs_ticket = True
    if issue_type in ["damaged_item", "payment_issue"]:
        needs_ticket = True
    if policy_result.get("eligible") is False:
        needs_ticket = True

    return {
        "needs_ticket": needs_ticket,
        "reason": "High priority / sensitive case / policy exception" if needs_ticket else "Auto-resolution is sufficient"
    }


def response_agent(customer_message: str, order_id: str, triage_result: Dict[str, Any], order_result: Dict[str, Any],
                   policy_result: Dict[str, Any], escalation_result: Dict[str, Any], ticket_result: Optional[Dict[str, Any]]) -> str:
    """
    Final customer-facing response writer.
    """
    system_prompt = (
        "You are a professional customer support responder. "
        "Write a concise, polite, solution-oriented response in clear English."
    )

    user_prompt = f"""
    Customer message: {customer_message}
    Order ID: {order_id}
    Triage result: {json.dumps(triage_result, indent=2)}
    Order result: {json.dumps(order_result, indent=2)}
    Policy result: {json.dumps(policy_result, indent=2)}
    Escalation result: {json.dumps(escalation_result, indent=2)}
    Ticket result: {json.dumps(ticket_result, indent=2) if ticket_result else "No ticket created"}

    Draft the final support response.
    """

    try:
        return call_llm(system_prompt, user_prompt, temperature=0.3)
    except Exception:
        # Fallback response if the API key is missing or the LLM call fails.
        base = f"I checked your request regarding '{triage_result['issue_type']}'. "
        base += f"Policy decision: {policy_result['decision']}. "
        if escalation_result["needs_ticket"] and ticket_result:
            base += f"A support ticket has been created with ID {ticket_result['ticket_id']}."
        else:
            base += "Your case can be handled automatically."
        return base

In [ ]:
# ============================================================
# MAIN MULTI-AGENT ORCHESTRATOR
# ============================================================

def run_customer_support_flow(customer_message: str, order_id: str) -> Dict[str, Any]:
    """
    Main pipeline that coordinates all agents.
    """
    triage_result = triage_agent(customer_message)
    order_result = order_agent(order_id)
    order_data = order_result["order"]

    policy_result = policy_agent(triage_result["issue_type"], order_data)
    escalation_result = escalation_agent(
        triage_result["issue_type"],
        triage_result["priority"],
        policy_result
    )

    ticket_result = None
    if escalation_result["needs_ticket"]:
        ticket_result = create_ticket(
            issue_type=triage_result["issue_type"],
            priority=triage_result["priority"],
            customer_message=customer_message,
            order_id=order_id,
        )

    final_response = response_agent(
        customer_message=customer_message,
        order_id=order_id,
        triage_result=triage_result,
        order_result=order_result,
        policy_result=policy_result,
        escalation_result=escalation_result,
        ticket_result=ticket_result,
    )

    return {
        "triage_result": triage_result,
        "order_result": order_result,
        "policy_result": policy_result,
        "escalation_result": escalation_result,
        "ticket_result": ticket_result,
        "final_response": final_response,
    }

In [ ]:
# ============================================================
# FASTAPI BACKEND
# ============================================================

app = FastAPI(title="Multi-Agent Customer Support System API")


class SupportRequest(BaseModel):
    customer_message: str = Field(..., description="Message entered by the customer")
    order_id: str = Field(default="", description="Order identifier if available")


@app.get("/health")
def health_check():
    return {"status": "ok", "message": "Customer Support API is running"}


@app.post("/triage")
def triage_endpoint(payload: SupportRequest):
    return triage_agent(payload.customer_message)


@app.post("/resolve")
def resolve_endpoint(payload: SupportRequest):
    return run_customer_support_flow(
        customer_message=payload.customer_message,
        order_id=payload.order_id,
    )


@app.get("/tickets")
def tickets_endpoint():
    return {"tickets": TICKETS_DB}

In [ ]:
# ============================================================
# API TESTING INSIDE COLAB
# ============================================================

test_client = TestClient(app)

sample_payload = {
    "customer_message": "My order was delivered but the item is damaged and I want a refund.",
    "order_id": "ORD1001"
}

# Uncomment to test after adding GROQ_API_KEY.
# response = test_client.post("/resolve", json=sample_payload)
# print(response.status_code)
# print(json.dumps(response.json(), indent=2))

In [ ]:
# ============================================================
# GRADIO UI
# ============================================================

def support_ui(customer_message, order_id):
    try:
        result = run_customer_support_flow(customer_message, order_id)

        triage_text = json.dumps(result["triage_result"], indent=2)
        policy_text = json.dumps(result["policy_result"], indent=2)
        final_text = result["final_response"]

        return triage_text, policy_text, final_text

    except Exception as e:
        return f"Error: {e}", "", ""


def ticket_table_ui():
    if not TICKETS_DB:
        return pd.DataFrame([{"message": "No tickets created yet"}])
    return pd.DataFrame(TICKETS_DB)


with gr.Blocks() as demo:
    gr.Markdown("# Multi-Agent Customer Support System")
    gr.Markdown("This demo simulates triage, order lookup, policy evaluation, escalation, and final response generation.")

    with gr.Row():
        customer_message = gr.Textbox(
            label="Customer Message",
            lines=5,
            placeholder="Example: My order was delayed and I want to know the status"
        )
        order_id = gr.Textbox(
            label="Order ID",
            placeholder="Example: ORD1002"
        )

    resolve_btn = gr.Button("Resolve Issue")

    triage_box = gr.Textbox(label="Triage Output", lines=10)
    policy_box = gr.Textbox(label="Policy Output", lines=10)
    final_box = gr.Textbox(label="Final Customer Response", lines=14)

    resolve_btn.click(
        fn=support_ui,
        inputs=[customer_message, order_id],
        outputs=[triage_box, policy_box, final_box]
    )

    gr.Markdown("## Ticket Log")
    ticket_btn = gr.Button("Refresh Tickets")
    ticket_table = gr.Dataframe()
    ticket_btn.click(fn=ticket_table_ui, inputs=None, outputs=ticket_table)

# Uncomment in Colab when you want the live UI.
# demo.launch(share=True)

## Final Notes
This notebook is strong for:
- multi-agent system explanation
- e-commerce support automation demos
- API architecture + UI in one notebook
- showing both deterministic rules and LLM-based handling

To run fully:
1. Add `GROQ_API_KEY` in Colab Secrets  
2. Run all cells  
3. Uncomment the test cell or `demo.launch(share=True)`